In [3]:
import pandas as pd

In [4]:
return_average = pd.read_csv("sac_average.csv", header = None)

In [5]:
return_average.rename(columns={0: "date", 1: "return"}, inplace=True)

In [6]:
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.sandwich_covariance import cov_hac

def newey_west_tstat(returns, maxlags=1):
    """
    논문 방식에 따른 Newey-West t-통계량 계산 함수
    입력:
        returns: 수익률 벡터 (list, np.array, pd.Series)
        maxlags: Newey-West 보정에 사용할 최대 시차
    출력:
        (평균 수익률, NW 표준오차, NW t-통계량)
    """
    returns = np.asarray(returns)
    T = len(returns)
    X = np.ones((T, 1))  # 상수항만 포함 (평균 추정)
    
    model = sm.OLS(returns, X).fit(cov_type='HAC', cov_kwds={'maxlags': maxlags})
    nw_cov = cov_hac(model, nlags=maxlags)
    # nw_se = np.sqrt(nw_cov[0, 0])
    # t_stat = model.params[0] / nw_se
    
    return model.params[0], model.bse[0], model.tvalues[0]


In [7]:

# 계산 실행
mean_return, nw_se, nw_tstat = newey_west_tstat(return_average["return"], maxlags=1)


In [8]:
from scipy.stats import t



# 단측 검정 (우측): P(T > t)
p_value = 1 - t.cdf(nw_tstat, df=len(return_average))

print(f"p-value = {p_value:.6f}")


p-value = 0.066830


In [9]:
nw_tstat

np.float64(1.5160306749105439)

In [10]:
return_average

,date,return
0,2019-01-30,0.030210
1,2019-02-28,0.006705
2,2019-03-28,-0.010164
3,2019-04-26,0.012148
4,2019-05-24,-0.020592
...,...,...
71,2024-09-20,0.032385
72,2024-10-18,0.004471
73,2024-11-15,-0.031152
74,2024-12-16,0.036831


In [11]:
np.quantile(return_average['return'], 0.05)

np.float64(-0.040597098)